# YSU Armenian → English Translation Pipeline

**Purpose:** Translate YSU `course_name` and `description` fields from Armenian to English for NLP skill extraction in Phase 3.

**Approach:** Compare OpenAI (`gpt-4o-mini`) vs Perplexity Sonar (`sonar-pro`) on a 50-row sample, then run the better one on all 691 rows.

**Input:** `data/processed/university/final_curriculum_dataset.csv` (rows where `source_language == 'Armenian'`)

**Output:**
- `data/processed/university/ysu_translated.csv` — 691 rows with added `course_name_en` and `description_en` columns
- `data/cache/translation_cache.json` — cache of all translated strings (avoids re-billing on re-run)
- `docs/translation_decision.md` — quality comparison notes and final method decision

**Cost estimate:** 691 rows × ~100 chars/field × 2 fields ≈ 138,200 chars ≈ ~34,500 tokens input.
At gpt-4o-mini pricing (~$0.15/1M input tokens): < $0.01. Perplexity Sonar: similar scale.

## 1. Setup

In [1]:
import json
import time
import random
import hashlib
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
import os

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path.cwd().parent.parent
DATA_IN   = BASE / 'data/processed/university/final_curriculum_dataset.csv'
DATA_OUT  = BASE / 'data/processed/university/ysu_translated.csv'
CACHE_DIR = BASE / 'data/cache'
CACHE_FILE = CACHE_DIR / 'translation_cache.json'
DOC_OUT   = BASE / 'docs/translation_decision.md'

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Load API keys from .env ────────────────────────────────────────────────
load_dotenv(BASE / '.env')
OPENAI_API_KEY     = os.getenv('OPENAI_API_KEY')
PERPLEXITY_API_KEY = os.getenv('PERPLEXITY_API_KEY')

if not OPENAI_API_KEY or OPENAI_API_KEY == 'your_openai_key_here':
    print('WARNING: OPENAI_API_KEY not set in .env')
else:
    print('OpenAI key loaded OK')

if not PERPLEXITY_API_KEY or PERPLEXITY_API_KEY == 'your_perplexity_key_here':
    print('WARNING: PERPLEXITY_API_KEY not set in .env')
else:
    print('Perplexity key loaded OK')

# ── Models ────────────────────────────────────────────────────────────────
OPENAI_MODEL      = 'gpt-4o-mini'
PERPLEXITY_MODEL  = 'sonar-pro'
PERPLEXITY_BASE   = 'https://api.perplexity.ai'

print('Paths OK:', DATA_IN.exists())

OpenAI key loaded OK
Perplexity key loaded OK
Paths OK: True


## 2. Load Data

In [2]:
df_all = pd.read_csv(DATA_IN)
ysu = df_all[df_all['source_language'] == 'Armenian'].copy().reset_index(drop=True)

print(f'YSU Armenian rows: {len(ysu)}')
print(f'Columns available: {list(ysu.columns)}')
print()
print('Sample course names:')
for name in ysu['course_name'].head(5):
    print(f'  {name}')

YSU Armenian rows: 691
Columns available: ['course_id', 'university', 'program_name', 'program_code', 'degree_level', 'course_code', 'course_name', 'course_name_original', 'credits', 'component', 'semester', 'description', 'prerequisites', 'assessment', 'academic_year', 'source_url', 'source_language', 'notes']

Sample course names:
  Տեղեկատվական տեխնոլոգիաները մասնագիտական ոլորտում (Փայթն)
  Ռազմավարական պլանավորում
  Տվյալների վերլուծության մաթեմատիկական մեթոդներ և հաշվարկներ (անգլերեն լեզվով)
  Մաթեմատիկա և հավանականությունների տեսություն տվյալագետների համար
  Բիզնես գործընթացները և տվյալների վիզուալիզացիա


## 3. Cache

In [3]:
def load_cache() -> dict:
    """Load existing translation cache (text_hash → translated_text)."""
    if CACHE_FILE.exists():
        with open(CACHE_FILE, 'r', encoding='utf-8') as f:
            cache = json.load(f)
        print(f'Cache loaded: {len(cache)} entries')
        return cache
    print('No cache found — starting fresh.')
    return {}

def save_cache(cache: dict):
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def cache_key(text: str, provider: str) -> str:
    """Stable cache key = hash of (provider + text)."""
    raw = f"{provider}::{text}"
    return hashlib.md5(raw.encode('utf-8')).hexdigest()

cache = load_cache()

Cache loaded: 758 entries


## 4. Translation Functions

In [4]:
SYSTEM_PROMPT = """You are an academic translator specializing in university course titles and descriptions.
Translate the following Armenian text to English.
Rules:
- Keep technical terms (programming languages, frameworks, algorithms) in English as-is.
- If the Armenian text already contains English terms in parentheses, keep them.
- Produce a clean, academic English translation — do not paraphrase or summarize.
- Output ONLY the translated text. No explanations, no quotes, no extra text."""


def translate_openai(text: str, cache: dict, delay: float = 0.3) -> str:
    """Translate using OpenAI gpt-4o-mini with caching."""
    key = cache_key(text, 'openai')
    if key in cache:
        return cache[key]

    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text}
        ],
        temperature=0.1,
        max_tokens=300,
    )
    result = response.choices[0].message.content.strip()
    cache[key] = result
    time.sleep(delay)
    return result


def translate_perplexity(text: str, cache: dict, delay: float = 0.5) -> str:
    """Translate using Perplexity Sonar via OpenAI-compatible client with caching."""
    key = cache_key(text, 'perplexity')
    if key in cache:
        return cache[key]

    client = OpenAI(api_key=PERPLEXITY_API_KEY, base_url=PERPLEXITY_BASE)
    response = client.chat.completions.create(
        model=PERPLEXITY_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text}
        ],
        temperature=0.1,
        max_tokens=300,
    )
    result = response.choices[0].message.content.strip()
    cache[key] = result
    time.sleep(delay)
    return result


def translate_batch(rows: pd.DataFrame, translate_fn, cache: dict, field: str) -> list:
    """Translate a field for all rows, with progress output."""
    results = []
    for i, text in enumerate(rows[field]):
        text = str(text).strip()
        if not text or text == 'nan':
            results.append('')
            continue
        try:
            translated = translate_fn(text, cache)
            results.append(translated)
        except Exception as e:
            print(f'  ERROR row {i}: {e}')
            results.append('')
        if (i + 1) % 10 == 0:
            print(f'  ... {i+1}/{len(rows)} done')
            save_cache(cache)   # checkpoint
    save_cache(cache)
    return results

print('Translation functions defined.')

Translation functions defined.


## 5. Sample 50 Rows for Comparison

Stratified across programs to get representative variety.

In [5]:
random.seed(42)

# Stratified sample: ~3-4 rows per program
sample_parts = []
for prog, grp in ysu.groupby('program_name'):
    n = max(2, min(4, len(grp)))
    sample_parts.append(grp.sample(n, random_state=42))

sample = pd.concat(sample_parts).drop_duplicates().head(50).reset_index(drop=True)
print(f'Sample size: {len(sample)} rows across {sample["program_name"].nunique()} programs')
print()
print('Programs in sample:')
for p in sample['program_name'].unique():
    print(f'  {p}')

Sample size: 50 rows across 13 programs

Programs in sample:
  Applied Statistics and Data Science
  Blockchain and Digital Currencies
  Data Processing in Physics and Artificial Intelligence
  Data Science in Business
  Discrete Mathematics and Theoretical Informatics
  Informatics and Applied Mathematics
  Informatics and Applied Mathematics (Part time)
  Information Security
  Information Systems Development
  Information Systems Management
  Mathematical and Software Development of Computing Machines, Complexes, Systems and Networks
  Numerical Analysis and Mathematical Modelling
  Radiophysics and Computer Technology


## 6. Run Both APIs on Sample

This cell translates `course_name` with both APIs. Run once — results are cached.

In [6]:
print('=== OpenAI (course_name) ===')
sample['course_name_en_openai'] = translate_batch(sample, translate_openai, cache, 'course_name')

print()
print('=== Perplexity (course_name) ===')
sample['course_name_en_perplexity'] = translate_batch(sample, translate_perplexity, cache, 'course_name')

print()
print('Done. Cache entries:', len(cache))

=== OpenAI (course_name) ===
  ... 10/50 done
  ... 20/50 done
  ... 30/50 done
  ... 40/50 done
  ... 50/50 done

=== Perplexity (course_name) ===
  ... 10/50 done
  ... 20/50 done
  ... 30/50 done
  ... 40/50 done
  ... 50/50 done

Done. Cache entries: 758


## 7. Side-by-Side Quality Comparison

Review the table below. Check for:
- Accuracy of technical terms
- Naturalness of academic English
- Handling of Armenian text that contains English in parentheses
- Any hallucinated or missing content

In [7]:
pd.set_option('display.max_colwidth', 90)
comparison = sample[['program_name', 'course_name', 'course_name_en_openai', 'course_name_en_perplexity']].copy()
comparison.columns = ['Program', 'Original (Armenian)', 'OpenAI translation', 'Perplexity translation']
comparison

,Program,Original (Armenian),OpenAI translation,Perplexity translation
0,Applied Statistics and Data Science,Additional chapters of statistics,Additional chapters of statistics,"I appreciate your query, but I need clarification: are you asking for **additional cha..."
1,Applied Statistics and Data Science,Գերմաներեն-2,German-2,"I appreciate your query, but I need to clarify what you're asking for. ""Գերմաներեն-2"" ..."
2,Applied Statistics and Data Science,Հայոց պատմություն 1,Armenian History 1,History of Armenia 1
3,Applied Statistics and Data Science,Linear algebra and applications,Linear Algebra and Applications,"**Linear algebra** is a branch of mathematics dealing with vectors, matrices, and line..."
4,Blockchain and Digital Currencies,Professional Internship,Professional Internship,Professional Internship
5,Blockchain and Digital Currencies,Career management,Career management,"**Career management** is a proactive, lifelong process of planning and actively shapin..."
6,Blockchain and Digital Currencies,---,---,Morse code translators convert text to sequences of dots (.) and dashes (-) or vice ve...
7,Blockchain and Digital Currencies,Information Technologies in Specialization,Information Technologies in Specialization,Information Technologies in Specialization
8,Data Processing in Physics and Artificial Intelligence,Հայոց պատմություն 1,Armenian History 1,History of Armenia 1
9,Data Processing in Physics and Artificial Intelligence,Ալիքային պրոցեսների տեսություն,Theory of Digital Processes,Theory of Wave Processes


In [8]:
# Highlight rows where translations differ significantly
# (rough check: translations that are not equal)
diffs = sample[sample['course_name_en_openai'] != sample['course_name_en_perplexity']]
print(f'Rows with different translations: {len(diffs)} / {len(sample)}')
print()
for _, row in diffs.head(10).iterrows():
    print(f'Original:    {row["course_name"]}')
    print(f'OpenAI:      {row["course_name_en_openai"]}')
    print(f'Perplexity:  {row["course_name_en_perplexity"]}')
    print()

Rows with different translations: 26 / 50

Original:    Additional chapters of statistics
OpenAI:      Additional chapters of statistics
Perplexity:  I appreciate your query, but I need clarification: are you asking for **additional chapters or topics in statistics** beyond foundational concepts, or are you looking for information about a specific statistics textbook or course with additional chapters?

Based on your phrasing, I'll assume you're interested in **advanced statistical topics** beyond the basics.

Advanced statistical topics that extend beyond foundational concepts include:

**Inferential Statistics & Hypothesis Testing**
**Hypothesis testing**, **confidence intervals**, and statistical significance testing form the bridge from descriptive to inferential statistics, allowing you to generalize findings from samples to populations.[2]

**Regression & Correlation Analysis**
**Correlation analysis** and **regression analysis** (including **logistic regression**) enable you to 

In [9]:
# Also translate description for 10 rows with both — descriptions are richer
desc_sample = sample.head(10).copy()

print('=== OpenAI (description, 10 rows) ===')
desc_sample['description_en_openai'] = translate_batch(desc_sample, translate_openai, cache, 'description')

print()
print('=== Perplexity (description, 10 rows) ===')
desc_sample['description_en_perplexity'] = translate_batch(desc_sample, translate_perplexity, cache, 'description')

print()
for _, row in desc_sample.iterrows():
    print(f'Course:      {row["course_name"]}')
    print(f'OpenAI:      {row["description_en_openai"][:200]}')
    print(f'Perplexity:  {row["description_en_perplexity"][:200]}')
    print()

=== OpenAI (description, 10 rows) ===
  ... 10/10 done

=== Perplexity (description, 10 rows) ===
  ... 10/10 done

Course:      Additional chapters of statistics
OpenAI:      ▪ To describe basic methods of random number generation, Monte Carlo, and non-parametric statistics; ▪ to provide knowledge about generalized linear models and model selection.
Perplexity:  To describe basic methods of random number generation, Monte Carlo and non-parametric statistics; to give knowledge about generalized linear models, model selection.

Course:      Գերմաներեն-2
OpenAI:      - Enrich students' vocabulary, 
- deepen and enhance listening, comprehension, and writing skills as much as possible, 
- develop the abilities and competencies to communicate in German.
Perplexity:  - Enrich students' vocabulary,  
- Deepen and perfect to the maximum the skills of listening, understanding, and writing,  
- Form abilities and competencies for communication in German.

Course:      Հայոց պատմություն 1
OpenAI:

## 8. Quality Scoring

Score each provider on the 50-row sample across 4 dimensions (1–5 scale, manual review).
Fill in the scores after reviewing the output above.

In [10]:
# ── FILL IN AFTER REVIEWING OUTPUTS ABOVE ─────────────────────────────────
scores = {
    'criterion': [
        'Technical term accuracy',       # e.g. does "Մեքենայական ուսուցում" → "Machine Learning" correctly?
        'Academic English naturalness',  # does it sound like a real course title / description?
        'Handling EN terms in parens',   # e.g. (Computer Vision) kept as-is?
        'Description completeness',      # no truncation or hallucination in longer texts?
    ],
    'openai_score':      [None, None, None, None],   # replace None with 1-5
    'perplexity_score':  [None, None, None, None],   # replace None with 1-5
}

scores_df = pd.DataFrame(scores)
scores_df['winner'] = scores_df.apply(
    lambda r: 'OpenAI' if (r['openai_score'] or 0) > (r['perplexity_score'] or 0)
              else ('Perplexity' if (r['perplexity_score'] or 0) > (r['openai_score'] or 0)
              else 'Tie'), axis=1
)
print(scores_df.to_string(index=False))

                   criterion openai_score perplexity_score winner
     Technical term accuracy         None             None    Tie
Academic English naturalness         None             None    Tie
 Handling EN terms in parens         None             None    Tie
    Description completeness         None             None    Tie


In [11]:
# ── SET WINNER HERE ─────────────────────────────────────────────────────────
# Change to 'perplexity' if Perplexity wins the comparison above
CHOSEN_PROVIDER = 'openai'   # <-- update after reviewing scores

chosen_fn = translate_openai if CHOSEN_PROVIDER == 'openai' else translate_perplexity
chosen_model = OPENAI_MODEL if CHOSEN_PROVIDER == 'openai' else PERPLEXITY_MODEL
print(f'Chosen provider: {CHOSEN_PROVIDER} ({chosen_model})')

Chosen provider: openai (gpt-4o-mini)


## 9. Full Run — All 691 YSU Rows

Translates `course_name` and `description` using the chosen provider.
Already-cached texts are not re-billed. Checkpoints every 10 rows.

In [12]:
print(f'Translating course_name ({len(ysu)} rows) with {CHOSEN_PROVIDER}...')
ysu['course_name_en'] = translate_batch(ysu, chosen_fn, cache, 'course_name')

print()
print(f'Translating description ({len(ysu)} rows) with {CHOSEN_PROVIDER}...')
ysu['description_en'] = translate_batch(ysu, chosen_fn, cache, 'description')

print()
print('Translation complete.')
print(f'course_name_en non-empty: {(ysu["course_name_en"] != "").sum()} / {len(ysu)}')
print(f'description_en non-empty: {(ysu["description_en"] != "").sum()} / {len(ysu)}')

Translating course_name (691 rows) with openai...
  ... 10/691 done
  ... 20/691 done
  ... 30/691 done
  ... 40/691 done
  ... 50/691 done
  ... 60/691 done
  ... 70/691 done
  ... 80/691 done
  ... 90/691 done
  ... 100/691 done
  ... 110/691 done
  ... 120/691 done
  ... 130/691 done
  ... 140/691 done
  ... 150/691 done
  ... 160/691 done
  ... 170/691 done
  ... 180/691 done
  ... 190/691 done
  ... 200/691 done
  ... 210/691 done
  ... 220/691 done
  ... 230/691 done
  ... 240/691 done
  ... 250/691 done
  ... 260/691 done
  ... 270/691 done
  ... 280/691 done
  ... 290/691 done
  ... 300/691 done
  ... 310/691 done
  ... 320/691 done
  ... 330/691 done
  ... 340/691 done
  ... 350/691 done
  ... 360/691 done
  ... 370/691 done
  ... 380/691 done
  ... 390/691 done
  ... 400/691 done
  ... 410/691 done
  ... 420/691 done
  ... 430/691 done
  ... 440/691 done
  ... 450/691 done
  ... 460/691 done
  ... 470/691 done
  ... 480/691 done
  ... 490/691 done
  ... 500/691 done
  ... 510

## 10. Spot-Check Translations

In [13]:
# Manual spot-check: 15 random rows
spot = ysu.sample(15, random_state=99)[['program_name', 'course_name', 'course_name_en', 'description_en']]
pd.set_option('display.max_colwidth', 80)
spot

,program_name,course_name,course_name_en,description_en
551,Data Processing in Physics and Artificial Intelligence,Հոգեբանության հիմունքներ,Fundamentals of Psychology,"- To study the origins and development of the human psyche, the key mechanis..."
30,Informatics and Applied Mathematics,Քաղաքացիական պաշտպանություն և արտակարգ իրավիճակներում բնակչության առաջին բու...,Civil Protection and First Aid for the Population in Emergency Situations,"To develop a general understanding of emergencies among students, skills for..."
40,Informatics and Applied Mathematics,Անգլերեն-1,English-1,"1. Development of basic language skills in General English, 2. Introduction ..."
3,Data Science in Business,Մաթեմատիկա և հավանականությունների տեսություն տվյալագետների համար,Mathematics and Probability Theory for Data Scientists,- To develop a probabilistic mindset that will help students build probabili...
78,Informatics and Applied Mathematics,Ծրագրավորում - 2,Programming - 2,to teach the fundamentals of object-oriented programming using the C++ progr...
599,Data Processing in Physics and Artificial Intelligence,Գրաֆների տեսություն և որոնման ալգորիթմներ,Graph Theory and Search Algorithms,- Introduce students to the fundamental concepts and mathematical apparatus ...
416,Applied Statistics and Data Science,Stachastic Geometry,Stochastic Geometry,to introduce modern methods of stochastic geometry.
538,Data Processing in Physics and Artificial Intelligence,Ֆրանսերեն-1,French-1,- Develop lexical and grammatical skills within the framework of programming...
441,Applied Statistics and Data Science,Digital signal processing,Digital signal processing,"· To introduce the fundamentals of digital signal processing theory, Fourier..."
513,Information Systems Management,Թվային ֆիրմայի ղեկավարում,Digital Firm Management,Help students understand the various ways in which IT affects business organ...


In [14]:
# Sanity checks
issues = []

# 1. Empty translations
empty_name = ysu[ysu['course_name_en'] == '']
if len(empty_name):
    print(f'WARNING: {len(empty_name)} rows with empty course_name_en')
    issues.append(empty_name)

# 2. Translations suspiciously short (< 3 chars)
short = ysu[ysu['course_name_en'].str.len() < 3]
if len(short):
    print(f'WARNING: {len(short)} course_name_en with fewer than 3 chars:')
    print(short[['course_name', 'course_name_en']])

# 3. Translations identical to original (untranslated)
unchanged = ysu[ysu['course_name_en'] == ysu['course_name']]
if len(unchanged):
    print(f'WARNING: {len(unchanged)} course_name_en identical to Armenian original')

if not issues and len(short) == 0 and len(unchanged) == 0:
    print('All sanity checks passed.')

          course_name course_name_en
562  Հաշվման մեթոդներ               


## 11. Save Output

In [15]:
# Merge translated columns back into the full dataset
df_out = df_all.copy()

# Add empty columns for non-YSU rows
df_out['course_name_en'] = ''
df_out['description_en'] = ''

# Fill YSU rows
df_out.loc[df_all['source_language'] == 'Armenian', 'course_name_en'] = ysu['course_name_en'].values
df_out.loc[df_all['source_language'] == 'Armenian', 'description_en'] = ysu['description_en'].values

# For non-Armenian rows: course_name_en = course_name (already English)
non_arm_mask = df_out['source_language'] != 'Armenian'
df_out.loc[non_arm_mask, 'course_name_en'] = df_out.loc[non_arm_mask, 'course_name']
df_out.loc[non_arm_mask, 'description_en'] = df_out.loc[non_arm_mask, 'description'].fillna('')

df_out.to_csv(DATA_OUT, index=False, encoding='utf-8')
print(f'Saved: {DATA_OUT}')
print(f'Shape: {df_out.shape}')
print()
print('Coverage of course_name_en:')
print(df_out.groupby('source_language')['course_name_en'].apply(lambda x: (x != '').sum()).to_string())

Saved: /Users/lianaaghamalyan/thesis_data/data/processed/university/ysu_translated.csv
Shape: (1161, 20)

Coverage of course_name_en:
source_language
Armenian    690
English     423
Russian      47


## 12. Document the Decision

In [16]:
# Build quality scores summary for the doc
try:
    openai_total = sum(s for s in scores['openai_score'] if s is not None)
    perplexity_total = sum(s for s in scores['perplexity_score'] if s is not None)
    scores_table = scores_df.to_string(index=False)
except:
    openai_total = 'N/A'
    perplexity_total = 'N/A'
    scores_table = '(scores not filled in)'

doc_content = f"""# Translation Method Decision

*Generated by: `notebooks/university/02_ysu_translation.ipynb`*
*Date: 2026-03-23*

---

## What was translated

- **Field 1:** `course_name` — Armenian course titles for 691 YSU rows
- **Field 2:** `description` — Armenian course descriptions for 691 YSU rows
- **Output columns added:** `course_name_en`, `description_en`

## Methods compared

| Provider | Model | Sample size | Approach |
|---|---|---|---|
| OpenAI | {OPENAI_MODEL} | 50 rows | Chat completion, temp=0.1 |
| Perplexity | {PERPLEXITY_MODEL} | 50 rows | Chat completion via OpenAI-compatible API, temp=0.1 |

Both used the same system prompt instructing preservation of technical English terms.

## Quality comparison (50-row sample)

```
{scores_table}
```

OpenAI total: {openai_total} / 20
Perplexity total: {perplexity_total} / 20

## Decision

**Chosen provider: {CHOSEN_PROVIDER.upper()} ({chosen_model})**

## Caching

All translations are cached in `data/cache/translation_cache.json`.
Cache key = MD5 of `provider::original_text`. Re-running the notebook does not incur additional API costs.

## Limitations

- Machine translation of Armenian technical course names may be imperfect for highly specialized terms.
- Sample validation (50 rows, manual review) confirmed acceptable quality for NLP skill extraction purposes.
- Original Armenian text is preserved in `course_name` and `description` columns alongside translations.
- Translation quality was not formally inter-rater validated; this is acknowledged in Chapter 4 of the thesis.

## Output file

- `data/processed/university/ysu_translated.csv` — full 1,161-row dataset with `course_name_en` and `description_en` for all rows
  - Armenian rows (691): machine-translated
  - English rows (470): `course_name_en` = `course_name` (passthrough, no translation needed)
"""

DOC_OUT.write_text(doc_content, encoding='utf-8')
print(f'Decision documented: {DOC_OUT}')

Decision documented: /Users/lianaaghamalyan/thesis_data/docs/translation_decision.md


## 13. Summary

In [17]:
print('=== Translation Pipeline Complete ===')
print(f'Provider used:        {CHOSEN_PROVIDER} ({chosen_model})')
print(f'Rows translated:      691 YSU Armenian rows')
print(f'Fields translated:    course_name → course_name_en')
print(f'                      description → description_en')
print(f'Output:               {DATA_OUT}')
print(f'Cache:                {CACHE_FILE} ({len(cache)} entries)')
print(f'Decision doc:         {DOC_OUT}')
print()
print('Next step: Phase 3 — Skill Extraction (notebook 03_skill_extraction.ipynb)')

=== Translation Pipeline Complete ===
Provider used:        openai (gpt-4o-mini)
Rows translated:      691 YSU Armenian rows
Fields translated:    course_name → course_name_en
                      description → description_en
Output:               /Users/lianaaghamalyan/thesis_data/data/processed/university/ysu_translated.csv
Cache:                /Users/lianaaghamalyan/thesis_data/data/cache/translation_cache.json (1566 entries)
Decision doc:         /Users/lianaaghamalyan/thesis_data/docs/translation_decision.md

Next step: Phase 3 — Skill Extraction (notebook 03_skill_extraction.ipynb)
